In [ ]:
import cv2
import torch
from transformers import ViTForImageClassification
from PIL import Image
import numpy as np
from torchvision import transforms

def load_model(repo_id="Engineer-Eslam/shape-recognizer-color", 
               device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Load the Vision Transformer model without preprocessor"""
    try:
        print(f"Loading model from Hugging Face: {repo_id}")
        print("Downloading model files...")
        
        # Load only the model (no preprocessor needed)
        model = ViTForImageClassification.from_pretrained(repo_id)
        
        model.to(device)
        model.eval()
        
        print("Model loaded successfully!")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

def create_manual_preprocessor():
    """Create manual preprocessing pipeline for ViT"""
    # Standard ViT preprocessing
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # ViT standard input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],  # Standard ImageNet normalization
            std=[0.5, 0.5, 0.5]
        )
    ])
    return transform

def preprocess_frame(frame, transform):
    """Preprocess frame for ViT model"""
    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Convert to PIL Image
    pil_image = Image.fromarray(rgb_frame)
    
    # Apply transformations
    tensor = transform(pil_image)
    
    # Add batch dimension
    tensor = tensor.unsqueeze(0)
    
    return tensor

def extract_shapes_from_frame(frame, min_area=1000):
    """Extract individual shapes from frame using contours"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )
    
    # Find contours
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    
    shape_regions = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        
        x, y, w, h = cv2.boundingRect(contour)
        
        # Add padding for better recognition
        padding = 20
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(frame.shape[1], x + w + padding)
        y2 = min(frame.shape[0], y + h + padding)
        
        # Make sure ROI has reasonable size
        if (x2 - x1) > 50 and (y2 - y1) > 50:
            shape_regions.append({
                'bbox': (x1, y1, x2, y2),
                'roi': frame[y1:y2, x1:x2],
                'contour': contour
            })
    
    return shape_regions, thresh

def main():
    # Setup device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Model repository ID
    REPO_ID = "Engineer-Eslam/shape-recognizer-color"
    
    # Load model
    model = load_model(REPO_ID, device)
    
    if model is None:
        print("\nFailed to load model.")
        return
    
    # Create manual preprocessor
    transform = create_manual_preprocessor()
    print("Manual preprocessor created")
    
    print(f"Number of classes: {model.config.num_labels}")
    
    # Get class labels if available
    if hasattr(model.config, 'id2label') and model.config.id2label:
        labels = model.config.id2label
        print(f"Classes detected: {labels}")
    else:
        # Default labels - adjust based on expected shapes
        labels = {
            0: "circle",
            1: "square", 
            2: "triangle",
            3: "rectangle",
            4: "pentagon",
            5: "hexagon",
            6: "star",
            7: "other"
        }
        print(f"Using default labels: {labels}")
    
    # Open webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open camera")
        return
    
    # Set camera properties for better performance
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    print("\n=== Real-Time Shape Recognition Started ===")
    print("\nControls:")
    print("  'q' - Quit")
    print("  'f' - Toggle full-frame mode")
    print("  's' - Toggle shape detection mode")
    print("  't' - Toggle threshold view")
    print("  '+' - Increase min area (filter smaller shapes)")
    print("  '-' - Decrease min area (detect smaller shapes)")
    
    full_frame_mode = False
    shape_mode = True
    show_thresh = False
    min_area = 1000
    
    # For FPS calculation
    import time
    prev_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        output = frame.copy()
        
        # Calculate FPS
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time)
        prev_time = curr_time
        
        if full_frame_mode:
            # Process entire frame
            input_tensor = preprocess_frame(frame, transform).to(device)
            
            with torch.no_grad():
                outputs = model(input_tensor)
                logits = outputs.logits
                probabilities = torch.softmax(logits, dim=1)
                confidence, predicted = torch.max(probabilities, 1)
                
                predicted_class = labels.get(predicted.item(), f"class_{predicted.item()}")
                conf_score = confidence.item()
            
            # Display prediction on frame
            text = f"{predicted_class}: {conf_score:.3f}"
            
            # Background for text
            (text_w, text_h), _ = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2
            )
            cv2.rectangle(output, (5, 5), (text_w + 15, text_h + 15), (0, 255, 0), -1)
            cv2.putText(output, text, (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
        
        elif shape_mode:
            # Extract and classify individual shapes
            shape_regions, thresh_img = extract_shapes_from_frame(frame, min_area)
            
            for region in shape_regions:
                x1, y1, x2, y2 = region['bbox']
                roi = region['roi']
                
                if roi.size == 0:
                    continue
                
                # Preprocess ROI for ViT
                input_tensor = preprocess_frame(roi, transform).to(device)
                
                # Run inference
                with torch.no_grad():
                    outputs = model(input_tensor)
                    logits = outputs.logits
                    probabilities = torch.softmax(logits, dim=1)
                    confidence, predicted = torch.max(probabilities, 1)
                    
                    predicted_class = labels.get(predicted.item(), f"class_{predicted.item()}")
                    conf_score = confidence.item()
                
                # Draw bounding box (color based on confidence)
                color = (0, 255, 0) if conf_score > 0.6 else (0, 165, 255) if conf_score > 0.3 else (0, 0, 255)
                cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
                
                # Prepare label
                label = f"{predicted_class}: {conf_score:.2f}"
                
                # Draw label background
                (label_w, label_h), _ = cv2.getTextSize(
                    label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2
                )
                cv2.rectangle(
                    output, (x1, y1 - label_h - 10), 
                    (x1 + label_w + 10, y1), color, -1
                )
                
                # Draw label text
                cv2.putText(
                    output, label, (x1 + 5, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2
                )
            
            # Display count and min area
            count_text = f"Shapes: {len(shape_regions)} | Min Area: {min_area}"
            cv2.putText(
                output, count_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2
            )
            
            # Show threshold if enabled
            if show_thresh:
                cv2.imshow("Threshold View", thresh_img)
        
        # Display FPS and mode info
        fps_text = f"FPS: {fps:.1f}"
        cv2.putText(
            output, fps_text, (10, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2
        )
        
        mode_text = "Full Frame" if full_frame_mode else "Shape Detection"
        cv2.putText(
            output, f"Mode: {mode_text}", (10, frame.shape[0] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2
        )
        
        # Show frame
        cv2.imshow("Shape Recognition - ViT", output)
        
        # Handle key presses
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('f'):
            full_frame_mode = not full_frame_mode
            shape_mode = False
            print(f"Full frame mode: {'ON' if full_frame_mode else 'OFF'}")
        elif key == ord('s'):
            shape_mode = not shape_mode
            full_frame_mode = False
            print(f"Shape detection mode: {'ON' if shape_mode else 'OFF'}")
        elif key == ord('t'):
            show_thresh = not show_thresh
            if not show_thresh:
                try:
                    cv2.destroyWindow("Threshold View")
                except:
                    pass
            print(f"Threshold view: {'ON' if show_thresh else 'OFF'}")
        elif key == ord('+') or key == ord('='):
            min_area += 200
            print(f"Min area increased to: {min_area}")
        elif key == ord('-'):
            min_area = max(200, min_area - 200)
            print(f"Min area decreased to: {min_area}")
    
    cap.release()
    cv2.destroyAllWindows()
    print("\nProgram terminated.")

if __name__ == "__main__":
    main()


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from transformers import ViTForImageClassification
from PIL import Image
import os
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import time

# FORCE NVIDIA GPU USAGE - CRITICAL!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Dataset Configuration
DATASET_PATH = r"" # PATH
TRAIN_DIR = os.path.join(DATASET_PATH, "train")

# Training Configuration
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
IMAGE_SIZE = 224
VALIDATION_SPLIT = 0.15
TEST_SPLIT = 0.15

# Model Configuration
BASE_MODEL = "google/vit-base-patch16-224"
OUTPUT_DIR = "./shape-classifier-vit-finetuned"

class ShapeDataset(Dataset):
    """Custom dataset for shape classification"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.idx_to_class = {idx: cls_name for cls_name, idx in self.class_to_idx.items()}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    img_path = os.path.join(class_dir, img_name)
                    self.samples.append((img_path, self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
        print(f"Classes: {self.classes}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color='white')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def create_transforms():
    """Create data augmentation transforms"""
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    return train_transform, val_transform

def split_train_val_test(dataset, val_split=0.15, test_split=0.15, random_seed=42):
    """Split dataset into train, validation, and test sets"""
    indices = list(range(len(dataset)))
    labels = [dataset.samples[i][1] for i in indices]
    
    train_val_indices, test_indices = train_test_split(
        indices, test_size=test_split, random_state=random_seed,
        stratify=labels, shuffle=True
    )
    
    train_val_labels = [labels[i] for i in train_val_indices]
    adjusted_val_split = val_split / (1 - test_split)
    
    train_indices, val_indices = train_test_split(
        train_val_indices, test_size=adjusted_val_split, random_state=random_seed,
        stratify=train_val_labels, shuffle=True
    )
    
    return train_indices, val_indices, test_indices

def train_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    """Train for one epoch with mixed precision"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training", ncols=100)
    for images, labels in pbar:
        # Move data to GPU - CRITICAL!
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs.logits, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * correct / total:.2f}%'
        })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    """Evaluate the model"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating", ncols=100)
        for images, labels in pbar:
            # Move data to GPU - CRITICAL!
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs.logits, labels)
            
            # Statistics
            running_loss += loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    epoch_loss = running_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    return epoch_loss, accuracy, precision, recall, f1

def main():
    print("=" * 80)
    print(" " * 20 + "ViT Shape Classifier Training - GPU Mode")
    print("=" * 80)
    
    # ========== GPU VERIFICATION ==========
    print("\n[1/6] Checking GPU Availability...")
    print("-" * 80)
    
    if not torch.cuda.is_available():
        print("❌ ERROR: CUDA is not available!")
        print("Please reinstall PyTorch with CUDA:")
        print("pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118")
        return
    
    device = torch.device('cuda:0')
    print(f"✓ PyTorch version: {torch.__version__}")
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"✓ Current Device Index: {torch.cuda.current_device()}")
    
    # Test GPU with a simple operation
    test_tensor = torch.randn(100, 100).to(device)
    print(f"✓ Test tensor on device: {test_tensor.device}")
    del test_tensor
    
    # ========== DATASET LOADING ==========
    print("\n[2/6] Loading Dataset...")
    print("-" * 80)
    
    if not os.path.exists(TRAIN_DIR):
        print(f"❌ ERROR: Dataset not found at {TRAIN_DIR}")
        return
    
    train_transform, val_transform = create_transforms()
    full_dataset = ShapeDataset(TRAIN_DIR, transform=None)
    
    # Split data
    train_indices, val_indices, test_indices = split_train_val_test(
        full_dataset, VALIDATION_SPLIT, TEST_SPLIT
    )
    
    print(f"\n✓ Dataset splits:")
    print(f"  • Training: {len(train_indices)} samples")
    print(f"  • Validation: {len(val_indices)} samples")
    print(f"  • Test: {len(test_indices)} samples")
    
    # Create datasets with transforms
    train_dataset = ShapeDataset(TRAIN_DIR, transform=train_transform)
    train_dataset = Subset(train_dataset, train_indices)
    
    val_dataset = ShapeDataset(TRAIN_DIR, transform=val_transform)
    val_dataset = Subset(val_dataset, val_indices)
    
    test_dataset = ShapeDataset(TRAIN_DIR, transform=val_transform)
    test_dataset = Subset(test_dataset, test_indices)
    
    # Create dataloaders with GPU optimization
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=0,  # Set to 0 for Windows
        pin_memory=True,  # Faster GPU transfer
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False
    )
    
    # ========== MODEL LOADING ==========
    print("\n[3/6] Loading Model...")
    print("-" * 80)
    
    temp_dataset = ShapeDataset(TRAIN_DIR, transform=None)
    num_classes = len(temp_dataset.classes)
    
    print(f"Loading {BASE_MODEL}...")
    model = ViTForImageClassification.from_pretrained(
        BASE_MODEL,
        num_labels=num_classes,
        id2label={i: temp_dataset.idx_to_class[i] for i in range(num_classes)},
        label2id={temp_dataset.idx_to_class[i]: i for i in range(num_classes)},
        ignore_mismatched_sizes=True
    )
    
    # MOVE MODEL TO GPU - MOST IMPORTANT STEP!
    model = model.to(device)
    
    print(f"✓ Model loaded with {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
    print(f"✓ Model is on device: {next(model.parameters()).device}")
    print(f"✓ Number of classes: {num_classes}")
    
    # ========== TRAINING SETUP ==========
    print("\n[4/6] Setting up Training...")
    print("-" * 80)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    
    # Mixed precision training for faster GPU training
    scaler = torch.cuda.amp.GradScaler()
    
    print(f"✓ Optimizer: AdamW (lr={LEARNING_RATE})")
    print(f"✓ Loss function: CrossEntropyLoss")
    print(f"✓ Mixed precision: Enabled")
    print(f"✓ Batch size: {BATCH_SIZE}")
    print(f"✓ Epochs: {NUM_EPOCHS}")
    
    # ========== TRAINING LOOP ==========
    print("\n[5/6] Training Model on GPU...")
    print("=" * 80)
    print("\n🚀 CHECK YOUR TASK MANAGER - GPU 1 USAGE SHOULD BE 80-95%!\n")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    best_val_acc = 0.0
    
    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
        print(f"{'='*80}")
        
        # Train
        start_time = time.time()
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler
        )
        train_time = time.time() - start_time
        
        print(f"\n📊 Training Results:")
        print(f"  • Loss: {train_loss:.4f}")
        print(f"  • Accuracy: {train_acc:.2f}%")
        print(f"  • Time: {train_time:.2f}s")
        
        # Validate
        val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(
            model, val_loader, criterion, device
        )
        
        print(f"\n📊 Validation Results:")
        print(f"  • Loss: {val_loss:.4f}")
        print(f"  • Accuracy: {val_acc*100:.2f}%")
        print(f"  • Precision: {val_prec:.4f}")
        print(f"  • Recall: {val_rec:.4f}")
        print(f"  • F1 Score: {val_f1:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
            }
            torch.save(checkpoint, os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"\n✓ ⭐ New best model saved! (Val Acc: {val_acc*100:.2f}%)")
    
    # ========== FINAL EVALUATION ==========
    print("\n[6/6] Final Evaluation on Test Set...")
    print("=" * 80)
    
    # Load best model
    checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate(
        model, test_loader, criterion, device
    )
    
    print(f"\n🎯 Final Test Results:")
    print(f"  • Accuracy: {test_acc*100:.2f}%")
    print(f"  • Precision: {test_prec:.4f}")
    print(f"  • Recall: {test_rec:.4f}")
    print(f"  • F1 Score: {test_f1:.4f}")
    
    # Save final model in HuggingFace format
    model.save_pretrained(OUTPUT_DIR)
    print(f"\n✓ Model saved to: {OUTPUT_DIR}")
    
    # Save class mapping
    with open(os.path.join(OUTPUT_DIR, 'class_mapping.txt'), 'w') as f:
        f.write("Class Mappings:\n")
        f.write("=" * 40 + "\n")
        for idx, class_name in temp_dataset.idx_to_class.items():
            f.write(f"{idx}: {class_name}\n")
    
    # Save training summary
    with open(os.path.join(OUTPUT_DIR, 'training_summary.txt'), 'w') as f:
        f.write("Training Summary\n")
        f.write("=" * 40 + "\n\n")
        f.write(f"Device: {device}\n")
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
        f.write(f"Epochs: {NUM_EPOCHS}\n")
        f.write(f"Batch Size: {BATCH_SIZE}\n")
        f.write(f"Learning Rate: {LEARNING_RATE}\n")
        f.write(f"Best Validation Accuracy: {best_val_acc*100:.2f}%\n")
        f.write(f"Final Test Accuracy: {test_acc*100:.2f}%\n")
        f.write(f"Test F1 Score: {test_f1:.4f}\n")
    
    print("\n" + "=" * 80)
    print("✅ Training Complete!")
    print("=" * 80)
    print(f"\n📁 All files saved to: {OUTPUT_DIR}")
    print("📊 You can now use this model for real-time inference!")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    print("\n🧹 GPU memory cleared")

if __name__ == "__main__":
    main()


In [ ]:
import accelerate
print(accelerate.__version__)


In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
import cv2
import torch
from transformers import ViTForImageClassification
from PIL import Image
from torchvision import transforms
import numpy as np
import time

# Configuration
MODEL_PATH = "./shape-classifier-vit-finetuned"
IMAGE_SIZE = 224
MIN_AREA = 1000
CONFIDENCE_THRESHOLD = 0.3

def load_model(model_path, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Load the trained ViT model"""
    try:
        print(f"Loading model from: {model_path}")
        model = ViTForImageClassification.from_pretrained(model_path)
        model = model.to(device)
        model.eval()
        
        print(f"✓ Model loaded successfully!")
        print(f"✓ Device: {device}")
        print(f"✓ Classes: {list(model.config.id2label.values())}")
        return model
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None

def create_transform():
    """Create preprocessing transform for ViT"""
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

def preprocess_frame(frame, transform):
    """Preprocess frame for ViT model"""
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_frame)
    tensor = transform(pil_image)
    tensor = tensor.unsqueeze(0)
    return tensor

def extract_shapes_from_frame(frame, min_area=1000):
    """Extract individual shapes from frame using contours"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )
    
    # Find contours
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    
    shape_regions = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        
        x, y, w, h = cv2.boundingRect(contour)
        
        # Add padding for better recognition
        padding = 20
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(frame.shape[1], x + w + padding)
        y2 = min(frame.shape[0], y + h + padding)
        
        # Make sure ROI has reasonable size
        if (x2 - x1) > 50 and (y2 - y1) > 50:
            shape_regions.append({
                'bbox': (x1, y1, x2, y2),
                'roi': frame[y1:y2, x1:x2],
                'contour': contour,
                'center': (x + w // 2, y + h // 2)
            })
    
    return shape_regions, thresh

def draw_prediction(frame, bbox, label, confidence, color):
    """Draw bounding box and label on frame"""
    x1, y1, x2, y2 = bbox
    
    # Draw bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    
    # Prepare label text
    text = f"{label}: {confidence:.2f}"
    
    # Calculate text size for background
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    thickness = 2
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    
    # Draw label background
    cv2.rectangle(
        frame,
        (x1, y1 - text_h - 10),
        (x1 + text_w + 10, y1),
        color,
        -1
    )
    
    # Draw label text
    cv2.putText(
        frame,
        text,
        (x1 + 5, y1 - 5),
        font,
        font_scale,
        (255, 255, 255),
        thickness
    )

def get_confidence_color(confidence):
    """Get color based on confidence level"""
    if confidence >= 0.7:
        return (0, 255, 0)  # Green - high confidence
    elif confidence >= 0.5:
        return (0, 255, 255)  # Yellow - medium confidence
    elif confidence >= 0.3:
        return (0, 165, 255)  # Orange - low confidence
    else:
        return (0, 0, 255)  # Red - very low confidence

def main():
    print("=" * 80)
    print(" " * 20 + "Real-Time Shape Detection with ViT")
    print("=" * 80)
    
    # Check GPU availability
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # Load trained model
    model = load_model(MODEL_PATH, device)
    if model is None:
        print("\n❌ Failed to load model. Make sure you've trained the model first!")
        print(f"Expected model path: {MODEL_PATH}")
        return
    
    # Get class labels
    if hasattr(model.config, 'id2label'):
        labels = model.config.id2label
    else:
        print("❌ Model doesn't have class labels!")
        return
    
    # Create transform
    transform = create_transform()
    
    # Open webcam
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Error: Could not open camera")
        return
    
    # Set camera properties
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    print("\n" + "=" * 80)
    print("🎥 Webcam opened successfully!")
    print("=" * 80)
    print("\n📋 Controls:")
    print("  [Q] - Quit")
    print("  [F] - Toggle full-frame mode (classify entire frame)")
    print("  [S] - Toggle shape detection mode (detect multiple shapes)")
    print("  [T] - Toggle threshold view (debug)")
    print("  [+] - Increase minimum area (filter smaller shapes)")
    print("  [-] - Decrease minimum area (detect smaller shapes)")
    print("  [C] - Toggle confidence threshold display")
    print("\n🎯 Starting detection...\n")
    
    # Mode flags
    full_frame_mode = False
    shape_mode = True
    show_thresh = False
    show_conf_threshold = True
    min_area = MIN_AREA
    
    # FPS calculation
    fps_time = time.time()
    fps = 0
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("❌ Failed to grab frame")
                break
            
            output = frame.copy()
            
            # Calculate FPS
            frame_count += 1
            if frame_count % 10 == 0:
                fps = 10 / (time.time() - fps_time)
                fps_time = time.time()
            
            if full_frame_mode:
                # Process entire frame
                input_tensor = preprocess_frame(frame, transform).to(device)
                
                with torch.no_grad():
                    outputs = model(input_tensor)
                    logits = outputs.logits
                    probabilities = torch.softmax(logits, dim=1)
                    confidence, predicted = torch.max(probabilities, 1)
                    
                    predicted_class = labels[predicted.item()]
                    conf_score = confidence.item()
                
                # Draw full-frame prediction
                color = get_confidence_color(conf_score)
                text = f"{predicted_class}: {conf_score:.3f}"
                
                font = cv2.FONT_HERSHEY_SIMPLEX
                (text_w, text_h), _ = cv2.getTextSize(text, font, 1.2, 3)
                
                cv2.rectangle(output, (10, 10), (text_w + 30, text_h + 30), color, -1)
                cv2.putText(output, text, (20, text_h + 20), font, 1.2, (255, 255, 255), 3)
                
            elif shape_mode:
                # Detect and classify individual shapes
                shape_regions, thresh_img = extract_shapes_from_frame(frame, min_area)
                
                detected_shapes = []
                
                for region in shape_regions:
                    x1, y1, x2, y2 = region['bbox']
                    roi = region['roi']
                    
                    if roi.size == 0:
                        continue
                    
                    # Preprocess ROI
                    input_tensor = preprocess_frame(roi, transform).to(device)
                    
                    # Run inference
                    with torch.no_grad():
                        outputs = model(input_tensor)
                        logits = outputs.logits
                        probabilities = torch.softmax(logits, dim=1)
                        confidence, predicted = torch.max(probabilities, 1)
                        
                        predicted_class = labels[predicted.item()]
                        conf_score = confidence.item()
                    
                    # Only show if confidence is above threshold
                    if conf_score >= CONFIDENCE_THRESHOLD:
                        color = get_confidence_color(conf_score)
                        draw_prediction(output, (x1, y1, x2, y2), predicted_class, conf_score, color)
                        detected_shapes.append(predicted_class)
                
                # Display shape count
                count_text = f"Shapes: {len(detected_shapes)} | Min Area: {min_area}"
                cv2.putText(output, count_text, (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                # Show threshold view if enabled
                if show_thresh:
                    cv2.imshow("Threshold View", thresh_img)
            
            # Display FPS
            fps_text = f"FPS: {fps:.1f}"
            cv2.putText(output, fps_text, (10, 70),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            
            # Display mode
            mode_text = "Mode: Full Frame" if full_frame_mode else "Mode: Shape Detection"
            cv2.putText(output, mode_text, (10, output.shape[0] - 50),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Display confidence threshold if enabled
            if show_conf_threshold:
                conf_text = f"Conf. Threshold: {CONFIDENCE_THRESHOLD:.2f}"
                cv2.putText(output, conf_text, (10, output.shape[0] - 20),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Display confidence legend
            legend_y = 110
            cv2.putText(output, "Confidence:", (10, legend_y),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.rectangle(output, (110, legend_y - 15), (140, legend_y - 5), (0, 255, 0), -1)
            cv2.putText(output, ">0.7", (145, legend_y - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            
            cv2.rectangle(output, (200, legend_y - 15), (230, legend_y - 5), (0, 255, 255), -1)
            cv2.putText(output, "0.5-0.7", (235, legend_y - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            
            cv2.rectangle(output, (310, legend_y - 15), (340, legend_y - 5), (0, 165, 255), -1)
            cv2.putText(output, "0.3-0.5", (345, legend_y - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            
            # Show frame
            cv2.imshow("Shape Detection - Your Trained ViT Model", output)
            
            # Handle keyboard input
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == ord('Q'):
                print("\n👋 Exiting...")
                break
            elif key == ord('f') or key == ord('F'):
                full_frame_mode = not full_frame_mode
                shape_mode = False
                print(f"{'✓' if full_frame_mode else '✗'} Full frame mode")
            elif key == ord('s') or key == ord('S'):
                shape_mode = not shape_mode
                full_frame_mode = False
                print(f"{'✓' if shape_mode else '✗'} Shape detection mode")
            elif key == ord('t') or key == ord('T'):
                show_thresh = not show_thresh
                if not show_thresh:
                    try:
                        cv2.destroyWindow("Threshold View")
                    except:
                        pass
                print(f"{'✓' if show_thresh else '✗'} Threshold view")
            elif key == ord('c') or key == ord('C'):
                show_conf_threshold = not show_conf_threshold
                print(f"{'✓' if show_conf_threshold else '✗'} Confidence threshold display")
            elif key == ord('+') or key == ord('='):
                min_area += 200
                print(f"Min area: {min_area}")
            elif key == ord('-') or key == ord('_'):
                min_area = max(200, min_area - 200)
                print(f"Min area: {min_area}")
    
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        print("\n" + "=" * 80)
        print("✅ Detection stopped. Webcam released.")
        print("=" * 80)

if __name__ == "__main__":
    main()
